# Economics Complete Experiment

Runs the full 20-seed economics CMDL, Plain LSTM, Grouped ARDL, and ablation suite, then saves table artifacts only.

In [ ]:
from argparse import Namespace
from pathlib import Path
import os
import shutil
import sys

import pandas as pd
from IPython.display import display

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

current = Path.cwd().resolve()
repo_root = next((p for p in [current, *current.parents] if (p / "experiments").exists() and (p / "config").exists()), None)
if repo_root is None:
    raise RuntimeError(f"Could not locate repo root from {current}")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from evaluation.economics_comparison import build_economics_comparison, build_mechanism_result_log
from evaluation.stratified_kstar import aggregate_per_method, build_economics_stratifiers, collect_seed_dirs, evaluate_method
from experiments import run_economics, run_economics_ablation, run_economics_ganet_baseline, run_economics_grouped_ardl, run_economics_lstm_baseline, run_economics_tft_baseline
from experiments.run_complete_20seed_suite import REALDATA_VARIANTS, cleanup, economics_common_args

PLAN_NAME = "complete_20seed_20260426"
SEEDS = list(range(20))
FORCE = False
RUN_CMDL = True
RUN_BASELINE = True
RUN_TFT = True
RUN_GANET = True
RUN_GROUPED_ARDL = True
RUN_ABLATIONS = True
N_PERM = 2000

OUTPUT_ROOT = repo_root / "outputs" / "notebook_economics" / PLAN_NAME
CMDL_DIR = OUTPUT_ROOT / "cmdl"
BASELINE_DIR = OUTPUT_ROOT / "plain_lstm"
TFT_DIR = OUTPUT_ROOT / "tft"
GANET_DIR = OUTPUT_ROOT / "ganet"
GROUPED_DIR = OUTPUT_ROOT / "grouped_ardl"
ABLATION_DIR = OUTPUT_ROOT / "ablation"
COMPARISON_DIR = OUTPUT_ROOT / "comparison"
for path in [CMDL_DIR, BASELINE_DIR, TFT_DIR, GANET_DIR, GROUPED_DIR, ABLATION_DIR, COMPARISON_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def summary_exists(run_dir: Path) -> bool:
    return (run_dir / "summary.json").exists()

def run_task(label: str, run_dir: Path, callback) -> None:
    if summary_exists(run_dir) and not FORCE:
        print(f"[skip] {label}: {run_dir}")
        return
    if run_dir.exists() and (FORCE or not summary_exists(run_dir)):
        reason = "force rerun" if FORCE else "incomplete artifact"
        print(f"[clean] {reason}: {run_dir}")
        shutil.rmtree(run_dir)
    print(f"[run] {label}: {run_dir}")
    callback()
    cleanup()
    if not summary_exists(run_dir):
        raise RuntimeError(f"Expected summary.json was not created for {label}: {run_dir}")

settings = pd.Series({"plan_name": PLAN_NAME, "seeds": SEEDS, "force": FORCE, "n_perm": N_PERM, "output_root": OUTPUT_ROOT}).to_frame("value")
display(settings)

c:\Users\42155\anaconda3\envs\PTenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,value
plan_name,complete_20seed_20260426
seeds,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
force,False
n_perm,2000
output_root,C:\DevSpace\PyDevspace\CMDL\outputs\notebook_e...


In [2]:
if RUN_CMDL:
    for seed in SEEDS:
        name = f"economics_cmdl_seed{seed}"
        args = Namespace(**economics_common_args(CMDL_DIR), seed=seed, experiment_name=name)
        run_task(f"economics CMDL seed {seed}", CMDL_DIR / name, lambda args=args: run_economics.run_experiment(args))

if RUN_BASELINE:
    for seed in SEEDS:
        name = f"economics_lstm_seed{seed}"
        args = Namespace(**economics_common_args(BASELINE_DIR), seed=seed, experiment_name=name)
        run_task(f"economics Plain LSTM seed {seed}", BASELINE_DIR / name, lambda args=args: run_economics_lstm_baseline.run_experiment(args))

if RUN_GROUPED_ARDL:
    for seed in SEEDS:
        name = f"economics_grouped_ardl_seed{seed}"
        args = Namespace(**economics_common_args(GROUPED_DIR), seed=seed, experiment_name=name)
        run_task(f"economics Grouped ARDL seed {seed}", GROUPED_DIR / name, lambda args=args: run_economics_grouped_ardl.run_experiment(args))

if RUN_ABLATIONS:
    ablation_args = Namespace(**economics_common_args(ABLATION_DIR), variant="all", seeds=SEEDS, experiment_prefix="economics_ablation")
    for seed in SEEDS:
        for variant in REALDATA_VARIANTS:
            name = f"economics_ablation_{variant}_seed{seed}"
            run_task(f"economics ablation {variant} seed {seed}", ABLATION_DIR / name, lambda variant=variant, seed=seed: run_economics_ablation.run_variant(ablation_args, variant, seed))

[run] economics CMDL seed 0: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_economics\complete_20seed_20260426\cmdl\economics_cmdl_seed0
[economics_cmdl_seed0] epoch=001 train_total=1.9567 val_task=2.9859 val_r2=-0.0789 val_proxy_r2=-0.3824
[economics_cmdl_seed0] epoch=010 train_total=0.6928 val_task=2.4848 val_r2=0.1126 val_proxy_r2=-0.3328
[economics_cmdl_seed0] epoch=020 train_total=0.5186 val_task=2.4850 val_r2=0.0937 val_proxy_r2=-0.2785
[economics_cmdl_seed0] epoch=030 train_total=0.4033 val_task=2.4506 val_r2=0.1068 val_proxy_r2=-0.2212
[economics_cmdl_seed0] early stopping at epoch 37
[run] economics CMDL seed 1: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_economics\complete_20seed_20260426\cmdl\economics_cmdl_seed1
[economics_cmdl_seed1] epoch=001 train_total=1.0464 val_task=2.9826 val_r2=-0.0697 val_proxy_r2=-0.2972
[economics_cmdl_seed1] epoch=010 train_total=0.6204 val_task=2.6055 val_r2=0.0650 val_proxy_r2=-0.2345
[economics_cmdl_seed1] epoch=020 train_total=0.4907 val_task

In [ ]:
if RUN_TFT:
    for seed in SEEDS:
        name = f"economics_tft_seed{seed}"
        args = Namespace(**economics_common_args(TFT_DIR), seed=seed, experiment_name=name)
        run_task(f"economics TFT seed {seed}", TFT_DIR / name, lambda args=args: run_economics_tft_baseline.run_experiment(args))

if RUN_GANET:
    for seed in SEEDS:
        name = f"economics_ganet_seed{seed}"
        args = Namespace(**economics_common_args(GANET_DIR), seed=seed, experiment_name=name)
        run_task(f"economics GA-Net seed {seed}", GANET_DIR / name, lambda args=args: run_economics_ganet_baseline.run_experiment(args))

In [ ]:
comparison = build_economics_comparison(
    cmdl_root=CMDL_DIR,
    baseline_root=BASELINE_DIR,
    tft_root=TFT_DIR,
    ganet_root=GANET_DIR,
    ablation_root=ABLATION_DIR,
    grouped_ardl_root=GROUPED_DIR,
)
mechanism_log = build_mechanism_result_log(comparison)

raw_csv = repo_root / "data" / "economics" / "processed" / "economics_cleaned_long_v2.csv"
stratifiers = build_economics_stratifiers(raw_csv)
method_dirs = {
    "CMDL": collect_seed_dirs(CMDL_DIR, "economics_cmdl_"),
    "Plain LSTM": collect_seed_dirs(BASELINE_DIR, "economics_lstm_"),
    "TFT": collect_seed_dirs(TFT_DIR, "economics_tft_"),
    "GA-Net": collect_seed_dirs(GANET_DIR, "economics_ganet_"),
    "No Recon Regularization": collect_seed_dirs(ABLATION_DIR, "economics_ablation_no_recon_regularization_"),
    "No AC Encoder": collect_seed_dirs(ABLATION_DIR, "economics_ablation_no_ac_encoder_"),
    "Uniform Lag": collect_seed_dirs(ABLATION_DIR, "economics_ablation_uniform_lag_"),
}
per_seed_frames = [evaluate_method(method, dirs, stratifiers, n_perm=N_PERM) for method, dirs in method_dirs.items() if dirs]
stratified_per_seed = pd.concat(per_seed_frames, ignore_index=True, sort=False) if per_seed_frames else pd.DataFrame()
stratified_aggregated = aggregate_per_method(stratified_per_seed) if not stratified_per_seed.empty else pd.DataFrame()

comparison.to_csv(COMPARISON_DIR / "economics_comparison.csv", index=False)
mechanism_log.to_csv(COMPARISON_DIR / "economics_mechanism_result_log.csv", index=False)
stratified_per_seed.to_csv(COMPARISON_DIR / "economics_stratified_kstar_per_seed.csv", index=False)
stratified_aggregated.to_csv(COMPARISON_DIR / "economics_stratified_kstar_aggregated.csv", index=False)

tables = {
    "economics_comparison": comparison,
    "economics_mechanism_result_log": mechanism_log,
    "economics_stratified_kstar_per_seed": stratified_per_seed,
    "economics_stratified_kstar_aggregated": stratified_aggregated,
}
pd.Series({name: len(frame) for name, frame in tables.items()}, name="rows").to_frame()

,rows
economics_comparison,120
economics_mechanism_result_log,6
economics_stratified_kstar_per_seed,240
economics_stratified_kstar_aggregated,12


In [4]:
for name, frame in tables.items():
    print(f"\n=== {name} ({len(frame)} rows) ===")
    display(frame.head(20))


=== economics_comparison (120 rows) ===


,family,display_name,experiment,model,variant,lag_method,tracking_backend,device,domain,scenario,...,test_effective_lag_mean,test_grouped_ardl_best_lag_mean,test_grouped_ardl_effective_lag_mean,test_grouped_ardl_group_count,test_grouped_ardl_low_best_lag,test_grouped_ardl_low_effective_lag,test_grouped_ardl_mid_best_lag,test_grouped_ardl_mid_effective_lag,test_grouped_ardl_high_best_lag,test_grouped_ardl_high_effective_lag
0,ablation,No AC Encoder,economics_ablation_no_ac_encoder_seed0,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,economics,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ablation,No AC Encoder,economics_ablation_no_ac_encoder_seed1,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,economics,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ablation,No AC Encoder,economics_ablation_no_ac_encoder_seed2,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,economics,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ablation,No AC Encoder,economics_ablation_no_ac_encoder_seed3,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,economics,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ablation,No AC Encoder,economics_ablation_no_ac_encoder_seed4,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,economics,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ablation,No AC Encoder,economics_ablation_no_ac_encoder_seed5,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,economics,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ablation,No AC Encoder,economics_ablation_no_ac_encoder_seed6,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,economics,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ablation,No AC Encoder,economics_ablation_no_ac_encoder_seed7,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,economics,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ablation,No AC Encoder,economics_ablation_no_ac_encoder_seed8,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,economics,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ablation,No AC Encoder,economics_ablation_no_ac_encoder_seed9,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,economics,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== economics_mechanism_result_log (6 rows) ===


,layer,question,answer,evidence
0,forecast_calibration,Does CMDL beat the matched LSTM?,no,"mean CMDL test_r2=0.05414579722609828, mean Pl..."
1,simple_baseline_calibration,Is CMDL above the simple calibrated baselines?,partial,"delta_vs_persistence=-0.8826874050038962, delt..."
2,ac_gate_mechanism,Does the anchor-adjusted lag-proxy direction s...,partial,"CMDL mean adjusted rho=-0.10869306317884078, p..."
3,ac_gate_per_proxy,Are all named proxy adjusted correlations alig...,partial,"candidate_positive_proxies=1/4, min_adjusted_r..."
4,ac_gate_heterogeneity,Is the learned lag gate non-degenerate?,yes,lag_gate_sensitivity_range=0.49912182092666624...
5,ablation_guard,Do degenerate controls expose the heterogeneit...,yes,"No AC kstar_std=0.0, Uniform Lag top1_share=1.0."



=== economics_stratified_kstar_per_seed (240 rows) ===


,method,seed,stratifier,n_entities,spearman_rho,perm_p_two_sided,kstar_std,degenerate
0,CMDL,0,log_gdp_per_worker_train,88,-0.603977,0.0000,0.033778,False
1,CMDL,0,hc_mean_train,88,-0.806224,0.0000,0.033778,False
2,CMDL,0,log_capital_per_worker_train,88,-0.564844,0.0000,0.033778,False
3,CMDL,1,log_gdp_per_worker_train,88,-0.348772,0.0020,0.022675,False
4,CMDL,1,hc_mean_train,88,-0.406273,0.0000,0.022675,False
5,CMDL,1,log_capital_per_worker_train,88,-0.279508,0.0110,0.022675,False
6,CMDL,2,log_gdp_per_worker_train,88,-0.424747,0.0000,0.224089,False
7,CMDL,2,hc_mean_train,88,-0.508453,0.0000,0.224089,False
8,CMDL,2,log_capital_per_worker_train,88,-0.416769,0.0000,0.224089,False
9,CMDL,3,log_gdp_per_worker_train,88,0.017911,0.8785,0.028368,False



=== economics_stratified_kstar_aggregated (12 rows) ===


,method,stratifier,n_seeds_total,n_seeds_valid,rho_mean,rho_median,abs_rho_mean,share_seeds_p_lt_05,share_seeds_p_lt_01,fisher_combined_p
0,CMDL,hc_mean_train,20,20,-0.141652,-0.295865,0.370599,0.80,0.75,1.009779e-46
1,CMDL,log_capital_per_worker_train,20,20,-0.099347,-0.135210,0.256791,0.65,0.50,1.382788e-24
2,CMDL,log_gdp_per_worker_train,20,20,-0.109463,-0.146094,0.278268,0.70,0.55,3.494823e-37
3,No AC Encoder,hc_mean_train,20,0,NaN,NaN,NaN,NaN,NaN,NaN
4,No AC Encoder,log_capital_per_worker_train,20,0,NaN,NaN,NaN,NaN,NaN,NaN
5,No AC Encoder,log_gdp_per_worker_train,20,0,NaN,NaN,NaN,NaN,NaN,NaN
6,No Recon Regularization,hc_mean_train,20,20,-0.218442,-0.298308,0.418462,0.80,0.65,1.533543e-50
7,No Recon Regularization,log_capital_per_worker_train,20,20,-0.112154,-0.123816,0.271589,0.60,0.45,6.137482e-30
8,No Recon Regularization,log_gdp_per_worker_train,20,20,-0.111099,-0.126035,0.286881,0.65,0.45,2.481622e-31
9,Uniform Lag,hc_mean_train,20,0,NaN,NaN,NaN,NaN,NaN,NaN
